In [0]:
pip install openpyxl

In [0]:
#Load Config and Setup Enviorment Variables
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()
 
# print(f"env_code: {lz_key}")  # This won't be redacted
# print(f"env_name: {env_name}")  # This won't be redacted
 
KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
# print(f"KeyVault_name: {KeyVault_name}")
 
# Service principal credentials
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")
 
# Storage account names
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"
external_storage = f"ingest{lz_key}external{env_name}"
  
# Spark config for curated storage (Delta table)
spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{external_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{external_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{external_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{external_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{external_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
  

In [0]:
# Requires openpyxl — add to cell 1 if not installed: %pip install openpyxl

import re
from pyspark.sql.functions import col, array, coalesce, lit, struct, date_format

# ── OUTPUT CONFIG ──────────────────────────────────────────────────────────────
LOCAL_PATH = "/tmp/gold_data_sheet.xlsx"        # driver-local, openpyxl writes here
DBFS_PATH  = "dbfs:/tmp/gold_data_sheet.xlsx"   # copy here to make downloadable

# ── TAB NAMES ─────────────────────────────────────────────────────────────────
TAB_NAMES = {
    "paymentPending":                "PP-CaseData",
    "appealSubmitted":               "AS-CaseData",
    "awaitingRespondentEvidence(a)": "AREa-CaseData",
    "awaitingRespondentEvidence(b)": "AREb-CaseData",
    "caseUnderReview":               "CUR-CaseData",
    "reasonsForAppealSubmitted":     "RFAS-CaseData",
    "listing":                       "LST-CaseData",
    "prepareForHearing":             "PFH-CaseData",
    "decision":                      "DEC-CaseData",
    "decided(a)":                    "DECa-CaseData",
    "decided(b)":                    "DECb-CaseData",
    "ftpaSubmitted(a)":              "FTSa-CaseData",
    "ftpaSubmitted(b)":              "FTSb-CaseData",
    "ftpaDecided":                   "FTPD-CaseData",
    "ended":                         "END-CaseData",
    "remitted":                      "REM-CaseData",
}

ALL_STATES = [
    "paymentPending",
    "appealSubmitted",
    "awaitingRespondentEvidence(a)",
    "awaitingRespondentEvidence(b)",
    "caseUnderReview",
    "reasonsForAppealSubmitted",
    "listing",
    "prepareForHearing",
    "decision",
    "decided(a)",
    "decided(b)",
    "ftpaSubmitted(a)",
    "ftpaSubmitted(b)",
    "ftpaDecided",
    "ended",
    "remitted"
]

# Priority columns appear first in every CaseData sheet; missing ones are skipped
PRIORITY_COLS = [
    "appealReferenceNumber", "TargetState", "dv_representation", "appealType",
    "appellantInUk", "submissionOutOfTime", "isRemissionsEnabled", "remissionType",
    "paymentStatus", "hasSponsor", "caseNotes",
]

# ── CCD AUDIT CONFIG ───────────────────────────────────────────────────────────
CCD_COLS = ["State", "AriaCaseNo", "CCDCaseID", "Status", "StartDateTime"]
CCD_PATH = (
    f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net"
    f"/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/all_active_states/ack_audit"
)
print("Loading CCD audit table…")
all_ccd_df = spark.read.format("delta").load(CCD_PATH)
print("CCD audit table loaded.")

# ── CDAM CASE NOTES CONFIG ─────────────────────────────────────────────────────
# Loaded once — covers all states; the per-state join inside the loop narrows it.
# dateAdded is formatted as a string in Spark (yyyy-MM-dd) so it serialises
# cleanly to JSON without going through Python's datetime.date repr.
CDAM_PATH = (
    f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net"
    f"/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/CDAM/ack_audit"
)
print("Loading CDAM case notes table…")
cdam_table = spark.read.format("delta").load(CDAM_PATH)

case_notes_df = cdam_table.select(
    col("CaseNo"),
    array(
        struct(
            struct(
                lit("Service Account").alias("user"),
                date_format(col("createdOn"), "yyyy-MM-dd").alias("dateAdded"),
                lit("ARIA Migrated Case").alias("caseNoteSubject"),
                lit("The attached document contains a copy of the original ARIA case data at the point of migration.").alias("caseNoteDescription"),
                struct(
                    col("document_url").alias("document_url"),
                    col("document_filename").alias("document_filename"),
                    col("document_binary_url").alias("document_binary_url")
                ).alias("caseNoteDocument")
            ).alias("value"),
            lit("1").alias("id")
        )
    ).alias("caseNotes")
)
print(f"CDAM case notes built: {case_notes_df.count()} entries across all states.")

import json as _json
from datetime import datetime
from openpyxl import Workbook
from openpyxl.utils import get_column_letter

RUN_DATE = datetime.now().strftime("%Y-%m-%d %H:%M")

def _row_to_dict(obj):
    """Recursively convert PySpark Row objects to plain Python dicts/lists
    so that json.dumps can serialise them without falling back to repr()."""
    if hasattr(obj, 'asDict'):
        return {k: _row_to_dict(v) for k, v in obj.asDict().items()}
    if isinstance(obj, list):
        return [_row_to_dict(item) for item in obj]
    if isinstance(obj, dict):
        return {k: _row_to_dict(v) for k, v in obj.items()}
    return obj

def _to_excel_val(v):
    """Convert a pandas cell value to something openpyxl can write.
    Nested structures (including PySpark Row objects) are serialised to
    indented JSON so the cell content is readable."""
    if v is None:
        return None
    if isinstance(v, (bool, int, float, str)):
        return v
    if str(v) in ("nan", "NaT", "None"):
        return None
    if isinstance(v, (dict, list)) or hasattr(v, 'asDict'):
        try:
            return _json.dumps(_row_to_dict(v), default=str, indent=2)
        except Exception:
            return str(v)
    return str(v)

def _autofit_columns(ws, min_width=10, max_width=100):
    """Set each column width to match its longest cell value (capped at max_width).
    Equivalent to selecting all columns and double-clicking the divider in Excel."""
    for col_cells in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col_cells[0].column)
        for cell in col_cells:
            if cell.value is not None:
                try:
                    max_len = max(max_len, len(str(cell.value)))
                except Exception:
                    pass
        ws.column_dimensions[col_letter].width = max(min_width, min(max_len + 2, max_width))

wb = Workbook()
wb.remove(wb.active)  # remove default empty sheet

# ── Loop Each State — creates two tabs per state (CaseData then CCD) ──────────
for state_under_test in ALL_STATES:
    print(f"\n{'='*60}")
    print(f"Processing: {state_under_test}")

    bronze_path = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
    silver_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
    gold_path   = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/{state_under_test}"

    # Get Latest Json Folder
    json_location        = dbutils.fs.ls(f"{gold_path}/")[-1]
    latest_json_location = json_location.name

    # Extract the date from the folder name (e.g. "2026-05-14/" -> "2026-05-14")
    _date_match = re.search(r'\d{4}-\d{2}-\d{2}', latest_json_location)
    gold_date = _date_match.group(0) if _date_match else latest_json_location.rstrip('/')
    print(f"  Gold date: {gold_date}")

    dbutils.fs.ls(f"{gold_path}/{latest_json_location}")

    # Set Paths
    try:
        json_path      = f"{gold_path}/{latest_json_location}/JSON/"
        M1_silver_path = f"{silver_path}/silver_appealcase_detail"
        M1_bronze_path = f"{bronze_path}/bronze_appealcase_crep_rep_floc_cspon_cfs"
        M2_silver_path = f"{silver_path}/silver_caseapplicant_detail"
        M3_silver_path = f"{silver_path}/silver_status_detail"
        C_path         = f"{silver_path}/silver_appealcategory_detail"
        bhc_path       = f"{bronze_path}/bronze_hearing_centres"
        bat_path       = f"{bronze_path}/bronze_appealtype"
        docsr_path     = f"{bronze_path}/bronze_documentsreceived"
    except Exception as e:
        print(f"Error during path setup: {str(e)}")

    # Create and Load Dataframes
    json_data = spark.read.format("json").load(json_path)
    M1_silver = spark.read.format("delta").load(M1_silver_path)
    M1_bronze = spark.read.format("delta").load(M1_bronze_path)
    M2_silver = spark.read.format("delta").load(M2_silver_path)
    M3_silver = spark.read.format("delta").load(M3_silver_path)
    C         = spark.read.format("delta").load(C_path)
    bhc       = spark.read.format("delta").load(bhc_path)
    bat       = spark.read.format("delta").load(bat_path)
    docsr     = spark.read.format("delta").load(docsr_path)

    from pyspark.sql.functions import (
        col, when, lit, array, struct, collect_list,
        max as spark_max, date_format, row_number, expr,
        size, udf, coalesce, concat_ws, concat, trim, year, split, datediff,
        collect_set, current_timestamp, transform, first, array_contains
    )

    # Patch missing witness interpreter fields for later-chain states
    if state_under_test in ("prepareForHearing", "decision", "decided(a)", "decided(b)", "ended", "remitted"):
        from pyspark.sql.types import StructType, StructField, StringType
        schema = StructType([
            StructField("appealReferenceNumber",              StringType(), True),
            StructField("witness1InterpreterSignLanguage",    StringType(), True),
            StructField("witness2InterpreterSignLanguage",    StringType(), True),
            StructField("witness3InterpreterSignLanguage",    StringType(), True),
            StructField("witness4InterpreterSignLanguage",    StringType(), True),
            StructField("witness5InterpreterSignLanguage",    StringType(), True),
            StructField("witness6InterpreterSignLanguage",    StringType(), True),
            StructField("witness7InterpreterSignLanguage",    StringType(), True),
            StructField("witness8InterpreterSignLanguage",    StringType(), True),
            StructField("witness9InterpreterSignLanguage",    StringType(), True),
            StructField("witness10InterpreterSignLanguage",   StringType(), True),
            StructField("witness1InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness2InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness3InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness4InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness5InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness6InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness7InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness8InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness9InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness10InterpreterSpokenLanguage", StringType(), True),
        ])
        missing_fields_json = spark.read.schema(schema).json(json_path)
        json_data = json_data.join(
            missing_fields_json,
            json_data["appealReferenceNumber"] == missing_fields_json["appealReferenceNumber"],
            "left"
        ).drop(missing_fields_json["appealReferenceNumber"])

    # ── Join silver representation and segmentation target state ──────────────
    M1_silver = M1_silver.select("CaseNo", "dv_representation")
    seg_df = spark.table("ariadm_active_appeals_bronze_silver.stg_segmentation_states").select("CaseNo", "TargetState")

    df  = json_data.join(M1_silver, M1_silver.CaseNo == json_data.appealReferenceNumber, "inner")
    df2 = df.join(seg_df, seg_df.CaseNo == df.appealReferenceNumber, "inner")
    df2 = df2.drop("CaseNo")  # drops both CaseNo columns added by the two joins

    # ── Replace empty gold caseNotes with real CDAM case notes ───────────────
    # Drop the placeholder empty array from gold, then left-join the real entries.
    # Left join keeps all gold records; coalesce gives [] for any case with no CDAM entry.
    if "caseNotes" in df2.columns:
        df2 = df2.drop("caseNotes")
    df2 = df2.join(case_notes_df, df2.appealReferenceNumber == case_notes_df.CaseNo, "left")
    df2 = df2.drop("CaseNo")
    df2 = df2.withColumn("caseNotes", coalesce(col("caseNotes"), array()))

    print(f"  Found: {df2.count()} records for {state_under_test}")

    # ── Order columns: priority fields first, then all remaining ──────────────
    existing_priority = [c for c in PRIORITY_COLS if c in df2.columns]
    ordered_cols = existing_priority + [c for c in df2.columns if c not in existing_priority]
    df2 = df2.select(*ordered_cols)

    # ── Tab 1: CaseData ───────────────────────────────────────────────────────
    tab_name = TAB_NAMES.get(state_under_test, state_under_test[:31])
    print(f"  Writing tab: {tab_name}")
    pdf = df2.toPandas()

    ws = wb.create_sheet(title=tab_name)
    ws["B2"] = "Last Updated"
    ws["C2"] = RUN_DATE
    ws["B3"] = "Gold Creation Date"
    ws["C3"] = gold_date

    for col_idx, col_name in enumerate(pdf.columns, start=2):
        ws.cell(row=5, column=col_idx, value=col_name)

    for row_idx, row_data in enumerate(pdf.itertuples(index=False), start=6):
        for col_idx, value in enumerate(row_data, start=2):
            ws.cell(row=row_idx, column=col_idx, value=_to_excel_val(value))

    _autofit_columns(ws)
    print(f"  {len(pdf)} rows, {len(pdf.columns)} columns written to '{tab_name}'")

    # ── Tab 2: CCD audit (immediately after CaseData tab for this state) ──────
    ccd_tab_name = tab_name.replace("-CaseData", "-CCD")
    print(f"  Writing tab: {ccd_tab_name}")

    ccd_state_df = (
        all_ccd_df
        .filter((col("State") == state_under_test) & (col("Status") != "ERROR"))
        .dropDuplicates(["AriaCaseNo"])
        .select(*[c for c in CCD_COLS if c in all_ccd_df.columns])
    )
    pdf_ccd = ccd_state_df.toPandas()

    # Convert CCDCaseID to string — long integers are misread by Excel as floats
    if "CCDCaseID" in pdf_ccd.columns:
        pdf_ccd["CCDCaseID"] = pdf_ccd["CCDCaseID"].apply(
            lambda x: None if (x is None or str(x) in ("nan", "NaT", "<NA>", "None")) else str(x)
        )

    ws_ccd = wb.create_sheet(title=ccd_tab_name)
    ws_ccd["B2"] = "Last Updated"
    ws_ccd["C2"] = RUN_DATE

    for col_idx, col_name in enumerate(pdf_ccd.columns, start=2):
        ws_ccd.cell(row=5, column=col_idx, value=col_name)

    # Locate CCDCaseID column so we can force number_format = '@' on each cell
    ccd_id_excel_col = (
        list(pdf_ccd.columns).index("CCDCaseID") + 2
        if "CCDCaseID" in pdf_ccd.columns else None
    )

    for row_idx, row_data in enumerate(pdf_ccd.itertuples(index=False), start=6):
        for col_idx, value in enumerate(row_data, start=2):
            cell = ws_ccd.cell(row=row_idx, column=col_idx, value=_to_excel_val(value))
            if col_idx == ccd_id_excel_col:
                cell.number_format = '@'

    _autofit_columns(ws_ccd)
    print(f"  {len(pdf_ccd)} rows written to '{ccd_tab_name}'")

# ── Save: write to driver /tmp first, then copy to DBFS ───────────────────────
# /dbfs/ FUSE mount does not support openpyxl's internal file operations (errno 95)
wb.save(LOCAL_PATH)
dbutils.fs.cp(f"file:{LOCAL_PATH}", DBFS_PATH)
print(f"\nWorkbook saved to DBFS: {DBFS_PATH}")
print("Download via the Databricks UI: Data > DBFS > tmp > gold_data_sheet.xlsx")

In [0]:
####################
#Copy to workspace
####################
import shutil
shutil.copy(
    "/tmp/gold_data_sheet.xlsx",
    "/Workspace/Users/peter.gresty@hmcts.net/gold_data_sheet.xlsx"
)